# Precious-Metals Statistical Arbitrage

## 1. Research question and decision standard

This notebook asks whether economically related precious-metals instruments exhibit a repeatable divergence/convergence effect that remains tradable after costs. It may conclude that no useful statistical arbitrage exists. Statistical significance is necessary, not sufficient, and none of the results guarantees future profit.

Verdicts are restricted to **Robust statistical-arbitrage evidence**, **Interesting but weak evidence**, or **No useful statistical arbitrage**.

## 2. Imports, deterministic configuration, and assumptions

Every historical estimate is point-in-time. The signal is observed first, execution occurs one session later, and P&L starts after execution. The base model charges 5 basis points per dollar of one-way turnover and 2% annualized short borrow.

The workflow follows the practical cautions in Michael Isichenko, *Quantitative Portfolio Management* (Wiley, 2021): prefer economically motivated relationships to brute-force pair mining, translate residual forecasts into actual tradable legs, model point-in-time information and costs, and avoid repeated P&L-driven tuning.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import adfuller, coint

from ats.ticker import EquityTicker

RANDOM_SEED = 20260926
TRADABLE_TICKERS = ("GLD", "IAU", "SLV", "GDX", "GDXJ", "SIL", "SILJ")
FACTOR_TICKERS = ("SPY", "UUP", "IEF", "TIP")


@dataclass(frozen=True)
class ResearchConfig:
    discovery_fraction: float = 0.60
    validation_fraction: float = 0.20
    entry_z: float = 2.0
    exit_z: float = 0.5
    max_holding_days: int = 20
    transaction_cost_bps: float = 5.0
    annual_short_borrow: float = 0.02
    bootstrap_repetitions: int = 2_000
    bootstrap_block_length: int = 40
    horizons: tuple[int, ...] = (1, 5, 10, 20, 40)
    estimation_windows: tuple[int, ...] = (252, 504)
    accumulation_windows: tuple[int, ...] = (5, 10, 20)
    normalization_windows: tuple[int, ...] = (126, 252)
    robustness_entry_z: tuple[float, ...] = (1.5, 2.0, 2.5)
    robustness_exit_z: tuple[float, ...] = (0.0, 0.5)
    robustness_holding_days: tuple[int, ...] = (10, 20, 40)
    robustness_cost_bps: tuple[float, ...] = (2.0, 5.0, 10.0)
    robustness_borrow: tuple[float, ...] = (0.0, 0.02, 0.05)


@dataclass(frozen=True)
class SplitBoundaries:
    discovery_end: pd.Timestamp
    validation_end: pd.Timestamp
    test_end: pd.Timestamp


CONFIG = ResearchConfig()
np.random.seed(RANDOM_SEED)

## 3. ATS API audit and data acquisition

Data are fetched below through `EquityTicker`; adjusted closes and actual observation dates are retained.

## 4. Data quality and chronological splits

Each candidate uses chronological 60/20/20 discovery, validation, and untouched-test partitions.

## 5. Reusable leakage-safe research functions

The tagged cells below contain the estimators, inference, and backtest machinery exercised by focused tests.

## 6. Deterministic self-tests

Assertions here complement the focused pytest suite with notebook-visible invariants.

## 7. Discovery diagnostics for candidate families

Only economically constrained cointegration, factor-residual, and rolling-PCA candidates are considered.

## 8. Convergence shape, robust inference, and rejection table

Forward convergence is tested at several horizons with overlap-aware inference and discovery-family multiplicity control.

## 9. Frozen strategy specification

The base entry, exit, holding period, sizing, timing, and cost assumptions are frozen before the untouched test is exposed.

## 10. Validation results and local robustness

Validation can reject the discovery candidate; robustness is diagnostic and never feeds parameter selection.

## 11. Untouched out-of-sample test

At most one validated primary candidate reaches this section.

## 12. Trading performance, regimes, and failure modes

Gross and net results, turnover, borrow, tails, drawdowns, subperiods, and lagged regimes are reported separately.

## 13. Concise final research summary

The executed notebook ends with the exact divergence, evidence, rule, out-of-sample result, robustness, limitations, and permitted verdict.